# 02 — Аудиошкала и транскрибация
## Цель
Построить единую временную шкалу для всех аудиофайлов книги и получить транскрипцию каждого файла через faster-whisper.

## Стратегия сохранения (защита от повторной транскрибации)
1. Сырые сегменты каждого файла сохраняются **сразу** после транскрибации в `raw_segments/` — с локальными таймкодами и всеми полями faster-whisper
2. Итоговый `audio_segments.pkl` (с глобальными таймкодами) — производный файл; его можно пересобрать из сырых сегментов + timeline
3. **Исправить шкалу, порядок файлов или пересчитать глобальные таймкоды можно без повторной транскрибации**

## Вход
- `data/besy/audio/*.mp3` — 23 аудиофайла в порядке воспроизведения

## Выход
- `raw_segments/` — по одному `.json` на аудиофайл: локальные таймкоды + все поля ASR (текст, avg_logprob, no_speech_prob, compression_ratio)
- `timeline.json` — карта глобальный ↔ локальный таймкод
- `audio_segments.pkl` — объединённые сегменты с глобальными таймкодами (для ноутбука 03)

In [1]:
import pickle, json, os
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get(
    "SPARK_ROOT",
    "/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit"
))

AUDIO_DIR = PROJECT_ROOT / "data/besy/audio"
OUTPUT_DIR = PROJECT_ROOT / "outputs/besy/run_01"
RAW_DIR = OUTPUT_DIR / "raw_segments"
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Собрать аудиофайлы в правильном порядке
audio_files = sorted(AUDIO_DIR.glob("*.mp3"))
print(f"Найдено файлов: {len(audio_files)}")
for i, f in enumerate(audio_files):
    print(f"  {i+1:2d}. {f.name}")

Найдено файлов: 24
   1. 101 Вместо введения - несколько подробностей из биографии многочтимого Степана Трофимовича Верховенского.mp3
   2. 102 Принц Гарри. Сватовство.mp3
   3. 103 Чужие грехи.mp3
   4. 104 Хромоножка.mp3
   5. 105 Премудрый змий.mp3
   6. 201 Ночь.mp3
   7. 202 Ночь (продолжение).mp3
   8. 203 Поединок.mp3
   9. 204 Все в ожидании.mp3
  10. 205 Пред праздником.mp3
  11. 206 Петр Степанович в хлопотах.mp3
  12. 207 У наших.mp3
  13. 208 Иван-Царевич.mp3
  14. 209 Степана Трофимовича описали.mp3
  15. 210 Флибустьеры. Роковое утро.mp3
  16. 301 Праздник. Отдел первый.mp3
  17. 302 Окончание праздника.mp3
  18. 303 Законченный роман.mp3
  19. 304 Последнее решение.mp3
  20. 305 Путешественница.mp3
  21. 306 Многотрудная ночь.mp3
  22. 307 Последнее странствование Степана Трофимовича.mp3
  23. 308 Заключение.mp3
  24. 309 У Тихона.mp3


In [3]:
# Длительность файлов через soundfile (без ffprobe)
import soundfile as sf

def get_duration(filepath):
    info = sf.info(str(filepath))
    return info.duration

print("OK: soundfile готов")

OK: soundfile готов


In [4]:
# Строим шкалу
timeline = []
global_offset = 0.0

for f in audio_files:
    duration = get_duration(f)
    timeline.append({
        "index": len(timeline),
        "file": f.name,
        "path": str(f),
        "duration": duration,
        "global_start": global_offset,
        "global_end": global_offset + duration,
    })
    global_offset += duration

total_duration = global_offset
h = int(total_duration // 3600)
m = int((total_duration % 3600) // 60)
print(f"Файлов: {len(timeline)} | Общая длительность: {h}ч {m}м ({total_duration:.0f}с)")
print()
for t in timeline:
    mm = int(t['duration'] // 60)
    ss = int(t['duration'] % 60)
    print(f"  [{t['index']:2d}] {t['file'][:55]:55s} {mm:3d}:{ss:02d}")

Файлов: 24 | Общая длительность: 36ч 12м (130333с)

  [ 0] 101 Вместо введения - несколько подробностей из биограф 114:03
  [ 1] 102 Принц Гарри. Сватовство.mp3                         126:37
  [ 2] 103 Чужие грехи.mp3                                     138:49
  [ 3] 104 Хромоножка.mp3                                      101:13
  [ 4] 105 Премудрый змий.mp3                                  146:46
  [ 5] 201 Ночь.mp3                                            142:16
  [ 6] 202 Ночь (продолжение).mp3                               77:59
  [ 7] 203 Поединок.mp3                                         37:10
  [ 8] 204 Все в ожидании.mp3                                   70:15
  [ 9] 205 Пред праздником.mp3                                  79:09
  [10] 206 Петр Степанович в хлопотах.mp3                      130:18
  [11] 207 У наших.mp3                                          71:30
  [12] 208 Иван-Царевич.mp3                                     29:51
  [13] 209 Степана Трофимовича описали

In [5]:
# Сохраняем timeline — это лёгкий файл, может понадобиться для пересчёта
with open(OUTPUT_DIR / "timeline.json", "w", encoding="utf-8") as f:
    json.dump(timeline, f, ensure_ascii=False, indent=2)
print("timeline.json сохранён")

timeline.json сохранён


In [6]:
# Транскрибация через faster-whisper
from faster_whisper import WhisperModel

MODEL_SIZE = "large-v3"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"
LANGUAGE = "ru"
BEAM_SIZE = 5

print(f"Загружаем faster-whisper {MODEL_SIZE} на {DEVICE} ({COMPUTE_TYPE})…")
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print("Модель загружена.")

Загружаем faster-whisper large-v3 на cuda (float16)…
Модель загружена.


In [7]:
from tqdm.notebook import tqdm

OVERWRITE = False

def segment_to_dict(seg):
    return {
        "start": round(seg.start, 3),
        "end": round(seg.end, 3),
        "text": seg.text.strip(),
        "avg_logprob": round(seg.avg_logprob, 4),
        "no_speech_prob": round(seg.no_speech_prob, 4),
        "compression_ratio": round(seg.compression_ratio, 4),
    }

def raw_path_for(t):
    safe_name = f"{t['index']:02d}_{t['file'].replace(' ', '_')[:80]}"
    return RAW_DIR / f"{safe_name}.json"

all_segments = []
seg_per_second = None

for t in timeline:
    idx = t["index"]
    rp = raw_path_for(t)
    
    if rp.exists() and not OVERWRITE:
        with open(rp, "r", encoding="utf-8") as f:
            saved = json.load(f)
        raw_segments = saved["segments"]
        print(f"[{idx+1}/{len(timeline)}] ПРОПУЩЕН — {t['file'][:60]} ({len(raw_segments)} сегм)")
    else:
        segments, info = model.transcribe(
            t["path"],
            language=LANGUAGE,
            beam_size=BEAM_SIZE,
            vad_filter=True,
        )
        
        if seg_per_second is not None:
            est_total = max(1, int(t["duration"] * seg_per_second))
        else:
            est_total = max(1, int(t["duration"] / 3.5))
        
        desc = f"[{idx+1}/{len(timeline)}] {t['file'][:50]}"
        raw_segments = []
        seg_bar = tqdm(total=est_total, desc=desc, unit="сг")
        for seg in segments:
            raw_segments.append(segment_to_dict(seg))
            seg_bar.update(1)
        seg_bar.close()
        
        actual = len(raw_segments)
        seg_per_second = actual / t["duration"]
        print(f"  -> завершён: {actual} сегментов")
        
        with open(rp, "w", encoding="utf-8") as f:
            json.dump({
                "file": t["file"],
                "path": t["path"],
                "duration": t["duration"],
                "model": MODEL_SIZE,
                "language": LANGUAGE,
                "segment_count": actual,
                "segments": raw_segments,
            }, f, ensure_ascii=False, indent=2)
    
    for seg in raw_segments:
        all_segments.append({
            "global_start": t["global_start"] + seg["start"],
            "global_end": t["global_start"] + seg["end"],
            "file": t["file"],
            "file_index": idx,
            "local_start": seg["start"],
            "local_end": seg["end"],
            "text": seg["text"],
            "avg_logprob": seg["avg_logprob"],
            "no_speech_prob": seg["no_speech_prob"],
            "compression_ratio": seg["compression_ratio"],
        })

print(f"\nВсего сегментов: {len(all_segments)}")
print(f"Средняя длительность сегмента: {total_duration / len(all_segments):.1f}с")

[1/24] ПРОПУЩЕН — 101 Вместо введения - несколько подробностей из биографии мн (3195 сегм)
[2/24] ПРОПУЩЕН — 102 Принц Гарри. Сватовство.mp3 (1598 сегм)


[3/24] 103 Чужие грехи.mp3:   0%|          | 0/2379 [00:00<?, ?сг/s]

  -> завершён: 1906 сегментов


[4/24] 104 Хромоножка.mp3:   0%|          | 0/1389 [00:00<?, ?сг/s]

  -> завершён: 1639 сегментов


...

  -> завершён: 1791 сегментов


[23/24] 308 Заключение.mp3:   0%|          | 0/605 [00:00<?, ?сг/s]

  -> завершён: 508 сегментов


[24/24] 309 У Тихона.mp3:   0%|          | 0/1494 [00:00<?, ?сг/s]

  -> завершён: 1567 сегментов

Всего сегментов: 35260
Средняя длительность сегмента: 3.7с


In [8]:
# Сохраняем итоговый файл с глобальными таймкодами
output_data = {
    "audio_segments": all_segments,
    "timeline": timeline,
    "total_duration": total_duration,
    "model": MODEL_SIZE,
    "language": LANGUAGE,
}

with open(OUTPUT_DIR / "audio_segments.pkl", "wb") as f:
    pickle.dump(output_data, f)

# Текстовая версия для ручной проверки
with open(OUTPUT_DIR / "transcript.txt", "w", encoding="utf-8") as f:
    for s in all_segments:
        mm = int(s["global_start"] // 60)
        ss = int(s["global_start"] % 60)
        f.write(f"[{mm:3d}:{ss:02d}] {s['text']}\n")

print(f"Сохранено в {OUTPUT_DIR}:")
print(f"  raw_segments/     — {len(timeline)} файлов (сырые, защита от перетранскрибации)")
print(f"  timeline.json     — шкала глобальных таймкодов")
print(f"  audio_segments.pkl — {len(all_segments)} сегментов (для ноутбука 03)")
print(f"  transcript.txt    — полная транскрипция (для ручной проверки)")

Сохранено в /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/run_01:
  raw_segments/     — 24 файлов (сырые, защита от перетранскрибации)
  timeline.json     — шкала глобальных таймкодов
  audio_segments.pkl — 35260 сегментов (для ноутбука 03)
  transcript.txt    — полная транскрипция (для ручной проверки)


In [9]:
# Быстрая проверка: примеры сегментов и их confidence
print("=== Первые 3 сегмента ===")
for s in all_segments[:3]:
    mm = int(s["local_start"] // 60)
    ss = int(s["local_start"] % 60)
    print(f"[{mm:2d}:{ss:02d}] logprob={s['avg_logprob']:.2f} no_speech={s['no_speech_prob']:.2f}")
    print(f"  {s['text'][:150]}")

print("\n=== Последние 3 сегмента ===")
for s in all_segments[-3:]:
    mm = int(s["local_start"] // 60)
    ss = int(s["local_start"] % 60)
    print(f"[{mm:2d}:{ss:02d}] logprob={s['avg_logprob']:.2f} no_speech={s['no_speech_prob']:.2f}")
    print(f"  {s['text'][:150]}")

# Статистика confidence
logprobs = [s["avg_logprob"] for s in all_segments]
no_speech = [s["no_speech_prob"] for s in all_segments]
print(f"\n=== Статистика confidence ===")
print(f"avg_logprob:    min={min(logprobs):.2f}  mean={sum(logprobs)/len(logprobs):.2f}  max={max(logprobs):.2f}")
print(f"no_speech_prob: min={min(no_speech):.2f}  mean={sum(no_speech)/len(no_speech):.2f}  max={max(no_speech):.2f}")

=== Первые 3 сегмента ===
[ 0:00] logprob=-0.16 no_speech=0.00
  Аудиоиздательство Вимбо представляет
[ 0:04] logprob=-0.16 no_speech=0.00
  Фёдор Михайлович Достоевский
[ 0:08] logprob=-0.16 no_speech=0.00
  Бесы

=== Последние 3 сегмента ===
[117:53] logprob=-0.16 no_speech=0.01
  Ставрогин даже задрожал от гнева и почти от испуга.
[117:59] logprob=-0.16 no_speech=0.01
  Проклятый психолог!
[118:03] logprob=-0.16 no_speech=0.01
  Оборвал он вдруг в бешенстве и, не оглядываясь, вышел из кельи.

=== Статистика confidence ===
avg_logprob:    min=-3.71  mean=-0.12  max=-0.02
no_speech_prob: min=0.00  mean=0.04  max=0.74


## Что делать, если нужно что-то поправить БЕЗ повторной транскрибации

### Исправить порядок файлов или глобальную шкалу
```python
# Загрузить сырые сегменты и timeline.json, пересчитать global_start/global_end
import json, pickle
from pathlib import Path

RAW_DIR = Path("outputs/besy/run_01/raw_segments")
with open("outputs/besy/run_01/timeline.json") as f:
    timeline = json.load(f)

# Поправить timeline (например, поменять порядок файлов)
# ...

# Пересобрать audio_segments.pkl
# ...
```

### Перетранскрибировать один проблемный файл
```python
# Удалить raw_segments/05_*.json, перезапустить только этот файл
```

## Проверка

1. Открыть `transcript.txt`
2. Взять 3–5 случайных таймкодов, открыть аудиофайл на этом времени
3. Сравнить текст транскрипции с тем, что слышно
4. Проверить, что стыки между файлами без разрывов/наложений
5. Обратить внимание на сегменты с высоким `no_speech_prob` (>0.5) — это тишина или музыка